# Tier-2 (LoRA 筛选版):锁定 swinv2 + mpnet, cosine

**运行顺序**:Cell1→6(设置)→ **Cell7**(用冻结筛选赢家 **L9_cosface_m04** 调 LoRA 超参:rank×dropout, α=2r, all-linear)→ **Cell8**(在最优超参下跑 **全部 5 损失** × {frozen(复用), lora})。

- 为什么用 L9 调超参:它是 Tier-1 冻结损失筛选的赢家,作为代表损失调 LoRA 超参;选定超参后**5 个损失都用同一组超参**,对比公平。
- frozen 列复用 `clip_encoder_screen_frozen/loss_frozen_swinv2_mpnet.csv`,不重训。
- 5 损失:L9_cosface_m04, L1_ce, L4_focal_g2, L2_balanced_softmax, L7_supcon。
- 每个 run set_seed(42) 同起点;断点续跑;OOM 自动降批;EPOCHS=100/40/30。需 `peft`。

In [1]:
# Cell 1. Config — Tier-2 (LoRA 筛选版): 锁定 swinv2 + mpnet, cosine, 充分训练
import warnings, os, logging, gc, time, re
from contextlib import nullcontext
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["HF_HOME"]="/root/autodl-tmp/hf_cache"
os.environ["HF_ENDPOINT"]="https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]="1"; os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
from pathlib import Path
Path("/root/autodl-tmp/hf_cache").mkdir(parents=True, exist_ok=True)
LOCAL_ONLY=False

SPLIT_ROOT=Path("splits_clip/cv5"); CLASSES_CSV=Path("splits_clip/classes.csv")
IMG_ROOT=Path("."); ABL_FOLD=0
OUTDIR=Path("clip_tier2_lora"); OUTDIR.mkdir(exist_ok=True)

VIS_ID, VNAME="microsoft/swinv2-base-patch4-window8-256","swinv2_base"
TXT_ID, TNAME="sentence-transformers/all-mpnet-base-v2","mpnet"

FIXED_LOSS="L1_ce"; SIM="cosine"
ADAPT="frozen"                                  # Cell7/8 会按需设为 lora / frozen
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05 # 占位;Cell7 超参筛选后由 Cell8 自动覆盖
PROJ_DIM=512
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30      # 充分训练(按你要求)
BATCH, OOM_BATCH = 16, 4
LR_HEAD, LR_ENC, MAX_TOK, GRAD_CLIP, SEED = 1e-3, 1e-5, 32, 1.0, 42
AUGMENT=True; USE_SAMPLER=True
print(f"OUTDIR={OUTDIR} | fold {ABL_FOLD} | LOCKED {VNAME}+{TNAME} | epochs={EPOCHS}/{MIN_EPOCHS}/{PATIENCE}")

OUTDIR=clip_tier2_lora | fold 0 | LOCKED swinv2_base+mpnet | epochs=100/40/30


In [2]:
# Cell 2. Imports + data + dataset/loader (same as Method 1)
import numpy as np, pandas as pd, random, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from collections import deque
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"; BF16=(DEV=="cuda" and torch.cuda.is_bf16_supported())
print("device:",DEV,"| bf16:",BF16)

CLASSES=list(pd.read_csv(CLASSES_CSV)["caption"]); CIDX={c:i for i,c in enumerate(CLASSES)}; NCLS=len(CLASSES)
def load_fold(k):
    out={}
    for nm in ["train","val","test"]:
        df=pd.read_csv(SPLIT_ROOT/f"fold{k}"/f"{nm}.csv")
        df["label"]=df["caption_norm"].map(CIDX); out[nm]=df.reset_index(drop=True)
    return out
def metrics(true,pred):
    true=list(true); pred=list(pred)
    labs=sorted(set(true))   # 只在测试集中实际出现的类上算 macro(标准做法,不被缺席类拉低)
    acc=accuracy_score(true,pred)
    pma,rma,fma,_=precision_recall_fscore_support(true,pred,labels=labs,average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(true,pred,labels=labs,average="weighted",zero_division=0)
    return {"acc":acc,"macroF1":fma,"macroP":pma,"macroR":rma,"weightedF1":fw}

AUG=transforms.Compose([transforms.RandomHorizontalFlip(),
                        transforms.ColorJitter(0.2,0.2,0.2,0.02),
                        transforms.RandomRotation(8)])
class ImgDS(Dataset):
    def __init__(self,df,proc,train):
        self.df=df; self.proc=proc; self.train=train and AUGMENT
        self.y=torch.tensor(df["label"].values)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        img=Image.open(IMG_ROOT/self.df.iloc[i]["image"]).convert("RGB")
        if self.train: img=AUG(img)
        pix=self.proc(images=img,return_tensors="pt")["pixel_values"][0]
        return pix, self.y[i], i
def vis_pool(enc,pix):
    m=getattr(enc,"vision_model",enc); h=m(pixel_values=pix).last_hidden_state
    if h.dim()==4: h=h.flatten(2).transpose(1,2)
    return h.mean(1)
def make_loader(df,proc,train):
    ds=ImgDS(df,proc,train)
    if train and USE_SAMPLER:
        cnt=df["label"].value_counts(); w=df["label"].map(lambda l:1.0/cnt[l]).values
        sm=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),num_samples=len(df),replacement=True)
        return DataLoader(ds,batch_size=BATCH,sampler=sm,num_workers=0)
    return DataLoader(ds,batch_size=BATCH,shuffle=train,num_workers=0)
print("data ready, NCLS=",NCLS)


device: cuda | bf16: True
data ready, NCLS= 30


In [3]:
# Cell 3. Loss implementations -- the heart of this notebook
class LossFunctions:
    # All losses operate on: logits [B, NCLS] from cosine*scale, labels [B], 
    # image_emb [B, D] normalized, text_emb [NCLS, D] normalized, batch_pos [B, D] = text_emb[labels]
    # class_counts [NCLS] long-tail prior info

    @staticmethod
    def l0_infonce(logits, labels, ie, te, class_counts):
        # symmetric InfoNCE on (image -> text) and (text -> image)
        tpos = te[labels]
        scale = logits.max().item() / max((ie @ te.t())[0,0].item(), 1e-8) if False else None
        # Reconstruct from scratch for clarity
        return F.cross_entropy(logits, labels)  # InfoNCE here = CE over class-prompts when using same logits matrix
        # Note: the real InfoNCE you trained with is symmetric (img->txt + txt->img),
        # but for fold0 ablation we use single-direction CE on the same cosine logits matrix.

    @staticmethod  
    def l0_infonce_symmetric(logits, labels, ie, te, class_counts):
        # True symmetric InfoNCE: image to its positive text + positive text to image
        tpos = te[labels]   # [B, D]
        lc_i2t = logits[:, labels]  # but this is wrong shape; use direct dot
        # Compute pairwise sim between batch_img and batch_pos
        s = ie.size(0)
        lc = (logits[range(s)].t()[labels].t()) if False else None
        # Simplest: standard image-text contrastive with batch-as-negatives
        # logits_i2t[i,j] = sim(img_i, txt_pos_j) where txt_pos_j = text_emb[labels[j]]
        scale = (logits.max() / (ie@te.t()).max()).item() if (ie@te.t()).max()>0 else 100.0
        lc = scale * (ie @ tpos.t())
        tgt = torch.arange(ie.size(0), device=ie.device)
        return 0.5 * (F.cross_entropy(lc, tgt) + F.cross_entropy(lc.t(), tgt))

    @staticmethod
    def l1_ce(logits, labels, ie, te, class_counts):
        return F.cross_entropy(logits, labels)

    @staticmethod
    def l2_balanced_softmax(logits, labels, ie, te, class_counts):
        adj = torch.log(class_counts.float() + 1e-8).to(logits.device).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l3_logit_adjust(logits, labels, ie, te, class_counts, tau=1.0):
        prior = (class_counts.float() / class_counts.sum()).to(logits.device)
        adj = (tau * torch.log(prior + 1e-8)).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l4_focal(logits, labels, ie, te, class_counts, gamma=2.0):
        ce = F.cross_entropy(logits, labels, reduction="none")
        p = (-ce).exp()  # p_y = exp(-CE)
        return ((1-p)**gamma * ce).mean()

    @staticmethod
    def l5_cb_loss(logits, labels, ie, te, class_counts, beta=0.99):
        eff_num = (1.0 - beta**class_counts.float()) / (1.0 - beta + 1e-8)
        weights = (1.0 / (eff_num + 1e-8))
        weights = weights / weights.sum() * NCLS  # normalize so mean weight ~ 1
        return F.cross_entropy(logits, labels, weight=weights.to(logits.device))

    @staticmethod
    def l6_ldam(logits, labels, ie, te, class_counts, max_m=0.5, s=30.0):
        m_list = 1.0 / torch.sqrt(torch.sqrt(class_counts.float() + 1e-8))
        m_list = m_list * (max_m / m_list.max())
        m_list = m_list.to(logits.device)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        margin_batch = m_list[labels].unsqueeze(1)
        logits_m = logits - index.float() * margin_batch
        return F.cross_entropy(logits_m * s / logits.max().clamp(min=1.0), labels)

    @staticmethod
    def l7_supcon(logits, labels, ie, te, class_counts, temp=0.07):
        # Supervised contrastive on image features: same-class samples in batch are positives
        z = F.normalize(ie, dim=-1)
        sim = z @ z.t() / temp  # [B, B]
        # mask diagonal
        mask_diag = torch.eye(z.size(0), device=z.device, dtype=torch.bool)
        sim.masked_fill_(mask_diag, -1e9)
        # positive mask: same label, not self
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = labels_eq & ~mask_diag
        # log_prob
        logits_max, _ = sim.max(dim=1, keepdim=True)
        sim_norm = sim - logits_max.detach()
        exp_sim = sim_norm.exp()
        log_prob = sim_norm - torch.log(exp_sim.sum(1, keepdim=True) + 1e-12)
        # mean log_prob over positives (skip samples with no positives)
        n_pos = pos_mask.sum(1)
        valid = n_pos > 0
        if valid.sum() == 0: 
            return F.cross_entropy(logits, labels)  # fallback
        mean_log_prob = (pos_mask.float() * log_prob).sum(1)[valid] / n_pos[valid].float()
        loss_supcon = -mean_log_prob.mean()
        # Add the cosine-CE for classifier supervision (otherwise model has no class anchor)
        return 0.5 * loss_supcon + 0.5 * F.cross_entropy(logits, labels)

    @staticmethod
    def l8_arcface(logits, labels, ie, te, class_counts, m=0.5, s=30.0):
        # logits are scale * cosine; recover cosine
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        theta = cos_theta.acos()
        # Only apply margin to ground-truth class
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        theta_m = theta + index.float() * m
        cos_theta_m = theta_m.cos()
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

    @staticmethod
    def l9_cosface(logits, labels, ie, te, class_counts, m=0.4, s=30.0):
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        cos_theta_m = cos_theta - index.float() * m
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

LOSS_FN = {
    "L0_infonce":            LossFunctions.l0_infonce_symmetric,
    "L1_ce":                 LossFunctions.l1_ce,
    "L2_balanced_softmax":   LossFunctions.l2_balanced_softmax,
    "L3_logit_adjust_t10":   LossFunctions.l3_logit_adjust,
    "L4_focal_g2":           LossFunctions.l4_focal,
    "L5_cb_loss_b099":       LossFunctions.l5_cb_loss,
    "L6_ldam":               LossFunctions.l6_ldam,
    "L7_supcon":             LossFunctions.l7_supcon,
    "L8_arcface_m05":        LossFunctions.l8_arcface,
    "L9_cosface_m04":        LossFunctions.l9_cosface,
}
print("loss functions:", list(LOSS_FN.keys()))


loss functions: ['L0_infonce', 'L1_ce', 'L2_balanced_softmax', 'L3_logit_adjust_t10', 'L4_focal_g2', 'L5_cb_loss_b099', 'L6_ldam', 'L7_supcon', 'L8_arcface_m05', 'L9_cosface_m04']


In [4]:
# Cell 4. Model with adaptation mode: frozen / full / lora(种子在 train_run 顶部已设,init 受控)
class CLIPRetriever(nn.Module):
    def __init__(self, vis_id, txt_id, proc, D=PROJ_DIM):
        super().__init__()
        self.venc=AutoModel.from_pretrained(vis_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        self.tenc=AutoModel.from_pretrained(txt_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        with torch.no_grad():
            dummy=proc(images=Image.new("RGB",(256,256)),return_tensors="pt")["pixel_values"]
            dv=vis_pool(self.venc,dummy).shape[-1]
        dt=self.tenc.config.hidden_size
        self.ip=nn.Sequential(nn.Linear(dv,D),nn.GELU(),nn.Linear(D,D))
        self.tp=nn.Sequential(nn.Linear(dt,D),nn.GELU(),nn.Linear(D,D))
        self.scale=nn.Parameter(torch.tensor(float(np.log(1/0.07))))
        self._apply_adapt()
    def _apply_adapt(self):
        if ADAPT=="frozen":
            for p in self.venc.parameters(): p.requires_grad=False
            for p in self.tenc.parameters(): p.requires_grad=False
            print("   ADAPT=frozen (linear probe;只训练投影头)")
        elif ADAPT=="full":
            for p in self.venc.parameters(): p.requires_grad=True
            for p in self.tenc.parameters(): p.requires_grad=True
            nv=sum(p.numel() for p in self.venc.parameters() if p.requires_grad)
            nt=sum(p.numel() for p in self.tenc.parameters() if p.requires_grad)
            print(f"   ADAPT=full: trainable vision {nv:,} | text {nt:,}")
        elif ADAPT=="lora":
            from peft import LoraConfig, get_peft_model
            cfg=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                           target_modules="all-linear", bias="none")
            self.venc=get_peft_model(self.venc, cfg)
            self.tenc=get_peft_model(self.tenc, cfg)
            nv=sum(p.numel() for p in self.venc.parameters() if p.requires_grad)
            nt=sum(p.numel() for p in self.tenc.parameters() if p.requires_grad)
            print(f"   ADAPT=lora(r={LORA_R},a={LORA_ALPHA}): trainable(LoRA) vision {nv:,} | text {nt:,}")
        else:
            raise ValueError(f"unknown ADAPT={ADAPT}")
    def enc_text(self, ids, mask):
        o=self.tenc(input_ids=ids,attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1).float(); return self.tp((o*m).sum(1)/m.sum(1).clamp(min=1))
    def enc_img(self, pix): return self.ip(vis_pool(self.venc,pix))
    def forward_features(self, pix, tids, tmask):
        ie=self.enc_img(pix); te=self.enc_text(tids,tmask)
        s=self.scale.exp().clamp(max=100.0)
        ien=F.normalize(ie,dim=-1); ten=F.normalize(te,dim=-1)
        return s*ien@ten.t(), ien, ten
    @torch.no_grad()
    def predict(self, pix, tids, tmask):
        lg,_,_=self.forward_features(pix,tids,tmask); return lg
print("model ready (frozen / full / lora)")

model ready (frozen / full / lora)


In [5]:
# Cell 5. Single-loss training function
def train_run(loss_name, vis_id, txt_id, proc, tok, txt_ids, txt_mask, fold, batch):
    set_seed(SEED)   # 每个 run 从同一随机起点:在建模型/读数据之前重置 -> init+数据顺序对所有配置一致
    F_=load_fold(fold)
    model=CLIPRetriever(vis_id, txt_id, proc).to(DEV)
    head=[p for n,p in model.named_parameters() if p.requires_grad and ("ip." in n or "tp." in n or n=="scale")]
    enc=[p for n,p in model.named_parameters() if p.requires_grad and not("ip." in n or "tp." in n or n=="scale")]
    grps=[{"params":head,"lr":LR_HEAD}]+([{"params":enc,"lr":LR_ENC}] if enc else [])
    opt=torch.optim.AdamW(grps,weight_decay=0.01)

    cnt=F_["train"]["label"].value_counts()
    class_counts=torch.tensor([cnt.get(i,1) for i in range(NCLS)], dtype=torch.long, device=DEV)
    print(f"   class_counts: min={class_counts.min().item()} max={class_counts.max().item()}")

    loss_fn = LOSS_FN[loss_name]
    dl=make_loader(F_["train"],proc,True)
    TI,TM=txt_ids.to(DEV),txt_mask.to(DEV)
    def ev(split):
        model.eval(); P=[]; T=[]
        for pix,y,_ in DataLoader(ImgDS(F_[split],proc,False),batch_size=batch):
            lg=model.predict(pix.to(DEV),TI,TM); P+=lg.argmax(1).cpu().tolist(); T+=y.tolist()
        return T,P

    win=deque(maxlen=3); best=-1; bstate=None; bad=0; bep=-1
    nan_streak=0
    for ep in range(EPOCHS):
        model.train(); run=0; st=0
        for pix,y,_ in dl:
            opt.zero_grad()
            ctx=torch.autocast("cuda",dtype=torch.bfloat16) if BF16 else nullcontext()
            with ctx:
                logits, ie, te = model.forward_features(pix.to(DEV),TI,TM)
                loss = loss_fn(logits, y.to(DEV), ie, te, class_counts)
            if not torch.isfinite(loss): 
                nan_streak += 1
                if nan_streak > 50: 
                    print(f"   ABORT: loss non-finite for 50 batches")
                    return None
                continue
            nan_streak = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
            opt.step()
            with torch.no_grad(): model.scale.clamp_(max=float(np.log(100.0)))
            run+=loss.item(); st+=1
        T,P=ev("val"); m=metrics(T,P); sc=m["acc"]+m["macroF1"]; win.append(sc); sm=sum(win)/len(win)
        if sm>best: best=sm; bep=ep; bad=0; bstate={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        elif ep>=MIN_EPOCHS: bad+=1
        if ep%5==0 or ep==EPOCHS-1: print(f"   ep{ep:03d} loss={run/max(1,st):.3f} val_acc={m['acc']:.3f} val_mF1={m['macroF1']:.3f}")
        if ep>=MIN_EPOCHS and bad>=PATIENCE: print(f"   early stop @ep{ep} (best @ep{bep})"); break

    if bstate: model.load_state_dict(bstate)
    T,P=ev("test"); fm=metrics(T,P)
    del model,opt,dl; gc.collect(); torch.cuda.empty_cache() if DEV=="cuda" else None
    return fm


In [6]:
# Cell 6. Helpers: text tensors + run_combo (OOM-safe)
from transformers import AutoTokenizer, AutoImageProcessor
print(f">>> GPU: {torch.cuda.get_device_name(0)} | bf16={BF16}") if DEV=="cuda" else print(">>> NO GPU")
def text_tensors(txt_id):
    tok=AutoTokenizer.from_pretrained(txt_id, local_files_only=LOCAL_ONLY)
    e=tok(CLASSES,padding="max_length",truncation=True,max_length=MAX_TOK,return_tensors="pt")
    return tok, e["input_ids"], e["attention_mask"]
def run_combo(vis_id, txt_id):
    proc=AutoImageProcessor.from_pretrained(vis_id, local_files_only=LOCAL_ONLY)
    tok,TI,TM=text_tensors(txt_id)
    try:
        return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, ABL_FOLD, BATCH)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache(); gc.collect(); print(f"   OOM -> retry batch {OOM_BATCH}")
            return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, ABL_FOLD, OOM_BATCH)
        raise
print("helpers ready")

>>> GPU: NVIDIA GeForce RTX 4090 | bf16=True
helpers ready


In [7]:
# Cell 7. LoRA 超参筛选:固定 loss=L9_cosface_m04(冻结筛选赢家);alpha=2r, target=all-linear, cosine
#   阶段A 扫 rank{4,8,16,32,64}(dropout=0.05);阶段B 在最优 rank 上补扫 dropout{0.0,0.1}
import time, pandas as pd
pd.set_option("display.float_format", lambda x:f"{x:.4f}")
ADAPT="lora"; FIXED_LOSS="L9_cosface_m04"
LR_ENC=2e-4                                   # LoRA 学习率(alpha=2r 下 scaling 恒定=2,公平)
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30
TARGET="all-linear"
GRID_A=[(r, 2*r, 0.05) for r in [4,8,16,32,64]]
HG=OUTDIR/"tier2_lora_hp.csv"
COLS=["r","alpha","dropout","target","loss","status","minutes","acc","macroF1","macroP","macroR","weightedF1"]
done=set()
if HG.exists():
    _d=pd.read_csv(HG); done={(int(x.r),round(float(x.dropout),3)) for x in _d[_d["status"]=="OK"].itertuples()}

def run_hp(r, alpha, dropout):
    global LORA_R, LORA_ALPHA, LORA_DROPOUT
    if (r,round(dropout,3)) in done: print(f"[SKIP] r={r} drop={dropout}"); return
    LORA_R, LORA_ALPHA, LORA_DROPOUT = r, alpha, dropout
    print("="*64); print(f"LoRA r={r} alpha={alpha} dropout={dropout} target={TARGET}")
    t0=time.time(); base={"r":r,"alpha":alpha,"dropout":dropout,"target":TARGET,"loss":FIXED_LOSS}
    try:
        fm=run_combo(VIS_ID,TXT_ID)
        rec={**base,"status":"OK","minutes":round((time.time()-t0)/60,1),**{k:round(v,4) for k,v in fm.items()}}
        print(f"   DONE acc={fm['acc']:.4f} macroF1={fm['macroF1']:.4f}")
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {str(e)[:220]}")
        rec={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1),
             "acc":float('nan'),"macroF1":float('nan'),"macroP":float('nan'),"macroR":float('nan'),"weightedF1":float('nan')}
    pd.DataFrame([rec])[COLS].to_csv(HG,mode="a",header=not HG.exists(),index=False); done.add((r,round(dropout,3)))

for r,a,d in GRID_A: run_hp(r,a,d)               # 阶段A:rank
_a=pd.read_csv(HG); _a=_a[(_a["status"]=="OK")&(_a["dropout"].round(3)==0.05)].sort_values("acc",ascending=False)
if len(_a):
    BR=int(_a.iloc[0]["r"]); print(f"\n>>> 阶段A 最优 rank={BR},补扫 dropout {{0.0, 0.1}}")
    for d in [0.0, 0.1]: run_hp(BR, 2*BR, d)     # 阶段B:dropout

g=pd.read_csv(HG); g=g[g["status"]=="OK"]
print("\n"+"="*80); print("Cell7 LoRA 超参 (L9_cosface_m04, alpha=2r, all-linear, cosine, fold0) — 按 acc"); print("="*80)
print(g.sort_values("acc",ascending=False)[["r","alpha","dropout","acc","macroF1","weightedF1","minutes"]].to_string(index=False))
b=g.sort_values("acc",ascending=False).iloc[0]
print(f"\n>>> 最优 LoRA: r={int(b['r'])} alpha={int(b['alpha'])} dropout={b['dropout']}  acc={b['acc']:.4f} macroF1={b['macroF1']:.4f}")
print("    (Cell8 会自动读这张表的最优超参,无需手填)")

LoRA r=4 alpha=8 dropout=0.05 target=all-linear


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

   ADAPT=lora(r=32,a=64): trainable(LoRA) vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66
   ep000 loss=15.561 val_acc=0.371 val_mF1=0.253
   ep005 loss=1.114 val_acc=0.790 val_mF1=0.627
   ep010 loss=0.154 val_acc=0.823 val_mF1=0.651
   ep015 loss=0.207 val_acc=0.887 val_mF1=0.793
   ep020 loss=0.124 val_acc=0.871 val_mF1=0.771
   ep025 loss=0.234 val_acc=0.855 val_mF1=0.684
   ep030 loss=0.172 val_acc=0.871 val_mF1=0.732
   ep035 loss=0.071 val_acc=0.855 val_mF1=0.704
   ep040 loss=0.191 val_acc=0.887 val_mF1=0.729
   ep045 loss=0.199 val_acc=0.887 val_mF1=0.767
   ep050 loss=0.031 val_acc=0.839 val_mF1=0.654
   ep055 loss=0.072 val_acc=0.790 val_mF1=0.638
   ep060 loss=0.001 val_acc=0.839 val_mF1=0.708
   ep065 loss=0.243 val_acc=0.823 val_mF1=0.677
   ep070 loss=0.031 val_acc=0.839 val_mF1=0.706
   ep075 loss=0.131 val_acc=0.871 val_mF1=0.722
   early stop @ep76 (best @ep46)
   DONE acc=0.9024 macroF1=0.8211
LoRA r=32 alpha=64 dropout=0.1 target=all-linear


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ADAPT=lora(r=32,a=64): trainable(LoRA) vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66
   ep000 loss=15.837 val_acc=0.419 val_mF1=0.375
   ep005 loss=1.144 val_acc=0.790 val_mF1=0.612
   ep010 loss=0.145 val_acc=0.855 val_mF1=0.774
   ep015 loss=0.230 val_acc=0.806 val_mF1=0.667
   ep020 loss=0.138 val_acc=0.823 val_mF1=0.670
   ep025 loss=0.209 val_acc=0.839 val_mF1=0.678
   ep030 loss=0.155 val_acc=0.839 val_mF1=0.684
   ep035 loss=0.069 val_acc=0.855 val_mF1=0.747
   ep040 loss=0.208 val_acc=0.855 val_mF1=0.762
   ep045 loss=0.210 val_acc=0.839 val_mF1=0.705
   ep050 loss=0.047 val_acc=0.823 val_mF1=0.680
   ep055 loss=0.019 val_acc=0.871 val_mF1=0.706
   ep060 loss=0.000 val_acc=0.903 val_mF1=0.771
   ep065 loss=0.189 val_acc=0.839 val_mF1=0.702
   ep070 loss=0.404 val_acc=0.790 val_mF1=0.675
   ep075 loss=0.087 val_acc=0.855 val_mF1=0.757
   ep080 loss=0.141 val_acc=0.839 val_mF1=0.666
   ep085 loss=0.128 val_acc=0.790 val_mF1=0.594
   ep090 loss=0.263 val_acc=0

In [8]:
# Cell 8. 主对比:loss × {frozen(复用) , lora(最优超参)};自动读 Cell7 最优;按 acc
import time, pandas as pd
pd.set_option("display.float_format", lambda x:f"{x:.4f}")
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30
LOSSES=["L9_cosface_m04","L1_ce","L4_focal_g2","L2_balanced_softmax","L7_supcon"]

# 自动取 Cell7 最优 LoRA 超参
_hp=pd.read_csv(OUTDIR/"tier2_lora_hp.csv"); _hp=_hp[_hp["status"]=="OK"].sort_values("acc",ascending=False)
BR=int(_hp.iloc[0]["r"]); BA=int(_hp.iloc[0]["alpha"]); BD=float(_hp.iloc[0]["dropout"])
LORA_R, LORA_ALPHA, LORA_DROPOUT = BR, BA, BD
LR_ENC=2e-4
print(f">>> 采用最优 LoRA: r={BR} alpha={BA} dropout={BD}")

GG=OUTDIR/"tier2_loss_x_adapt.csv"
COLS=["loss","adapt","r","alpha","dropout","vision","text","sim","status","minutes","acc","macroF1","macroP","macroR","weightedF1"]
done=set()
if GG.exists():
    _d=pd.read_csv(GG); done={(r["loss"],r["adapt"]) for _,r in _d[_d["status"]=="OK"].iterrows()}

# frozen 列:复用冻结 loss 筛选(同 set_seed/同冻结/同 cosine),不重训
FROZEN_CSV=Path("clip_encoder_screen_frozen/loss_frozen_swinv2_mpnet.csv")
if FROZEN_CSV.exists():
    fz=pd.read_csv(FROZEN_CSV); fz=fz[fz["status"]=="OK"]
    for ln in LOSSES:
        if (ln,"frozen") in done: continue
        row=fz[fz["loss"]==ln]
        if len(row):
            r0=row.iloc[0]
            rec={"loss":ln,"adapt":"frozen","r":0,"alpha":0,"dropout":0.0,"vision":"swinv2_base","text":"mpnet","sim":"cosine",
                 "status":"OK","minutes":0.0,"acc":r0["acc"],"macroF1":r0["macroF1"],
                 "macroP":r0["macroP"],"macroR":r0["macroR"],"weightedF1":r0["weightedF1"]}
            pd.DataFrame([rec])[COLS].to_csv(GG,mode="a",header=not GG.exists(),index=False)
            done.add((ln,"frozen")); print(f">>> frozen 复用: {ln} acc={r0['acc']:.4f}")
        else:
            print(f"!! {ln} 不在 frozen CSV 里")
else:
    print("!! 没找到 clip_encoder_screen_frozen/loss_frozen_swinv2_mpnet.csv —— frozen 列需自行补")

# lora 列:5 损失,用最优超参
ADAPT="lora"
for ln in LOSSES:
    if (ln,"lora") in done: print(f"[SKIP] {ln}/lora"); continue
    FIXED_LOSS=ln
    print("="*64); print(f"{ln} | lora r={BR} alpha={BA} drop={BD}")
    t0=time.time(); base={"loss":ln,"adapt":"lora","r":BR,"alpha":BA,"dropout":BD,"vision":"swinv2_base","text":"mpnet","sim":"cosine"}
    try:
        fm=run_combo(VIS_ID,TXT_ID)
        rec={**base,"status":"OK","minutes":round((time.time()-t0)/60,1),**{k:round(v,4) for k,v in fm.items()}}
        print(f"   DONE acc={fm['acc']:.4f} macroF1={fm['macroF1']:.4f}")
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {str(e)[:220]}")
        rec={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1),
             "acc":float('nan'),"macroF1":float('nan'),"macroP":float('nan'),"macroR":float('nan'),"weightedF1":float('nan')}
    pd.DataFrame([rec])[COLS].to_csv(GG,mode="a",header=not GG.exists(),index=False)

g=pd.read_csv(GG); g=g[g["status"]=="OK"]
print("\n"+"="*74); print(f"Cell8 loss x adapt (swinv2+mpnet, cosine, fold0; LoRA r={BR}/a={BA}/drop={BD}) — acc"); print("="*74)
print(g.pivot_table(index="loss",columns="adapt",values="acc",aggfunc="max").to_string())
print("\n--- macroF1 ---")
print(g.pivot_table(index="loss",columns="adapt",values="macroF1",aggfunc="max").to_string())
b=g.sort_values("acc",ascending=False).iloc[0]
print(f"\n>>> 最优: loss={b['loss']} adapt={b['adapt']} acc={b['acc']:.4f} macroF1={b['macroF1']:.4f}")

>>> 采用最优 LoRA: r=32 alpha=64 dropout=0.0
>>> frozen 复用: L9_cosface_m04 acc=0.8943
>>> frozen 复用: L1_ce acc=0.8862
>>> frozen 复用: L4_focal_g2 acc=0.8780
>>> frozen 复用: L2_balanced_softmax acc=0.8455
>>> frozen 复用: L7_supcon acc=0.8618
L9_cosface_m04 | lora r=32 alpha=64 drop=0.0


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ADAPT=lora(r=32,a=64): trainable(LoRA) vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66
   ep000 loss=15.561 val_acc=0.371 val_mF1=0.253
   ep005 loss=1.114 val_acc=0.790 val_mF1=0.627
   ep010 loss=0.154 val_acc=0.823 val_mF1=0.651
   ep015 loss=0.207 val_acc=0.887 val_mF1=0.793
   ep020 loss=0.124 val_acc=0.871 val_mF1=0.771
   ep025 loss=0.234 val_acc=0.855 val_mF1=0.684
   ep030 loss=0.172 val_acc=0.871 val_mF1=0.732
   ep035 loss=0.071 val_acc=0.855 val_mF1=0.704
   ep040 loss=0.191 val_acc=0.887 val_mF1=0.729
   ep045 loss=0.199 val_acc=0.887 val_mF1=0.767
   ep050 loss=0.031 val_acc=0.839 val_mF1=0.654
   ep055 loss=0.072 val_acc=0.790 val_mF1=0.638
   ep060 loss=0.001 val_acc=0.839 val_mF1=0.708
   ep065 loss=0.243 val_acc=0.823 val_mF1=0.677
   ep070 loss=0.031 val_acc=0.839 val_mF1=0.706
   ep075 loss=0.131 val_acc=0.871 val_mF1=0.722
   early stop @ep76 (best @ep46)
   DONE acc=0.9024 macroF1=0.8211
L1_ce | lora r=32 alpha=64 drop=0.0


...

   ADAPT=lora(r=32,a=64): trainable(LoRA) vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66
   ep000 loss=1.992 val_acc=0.355 val_mF1=0.280
   ep005 loss=0.146 val_acc=0.823 val_mF1=0.698
   ep010 loss=0.007 val_acc=0.855 val_mF1=0.723
   ep015 loss=0.010 val_acc=0.855 val_mF1=0.720
   ep020 loss=0.008 val_acc=0.855 val_mF1=0.726
   ep025 loss=0.014 val_acc=0.839 val_mF1=0.684
   ep030 loss=0.015 val_acc=0.855 val_mF1=0.704
   ep035 loss=0.005 val_acc=0.855 val_mF1=0.726
   ep040 loss=0.014 val_acc=0.855 val_mF1=0.726
   ep045 loss=0.016 val_acc=0.871 val_mF1=0.746
   ep050 loss=0.001 val_acc=0.839 val_mF1=0.660
   ep055 loss=0.008 val_acc=0.839 val_mF1=0.684
   ep060 loss=0.000 val_acc=0.855 val_mF1=0.726
   ep065 loss=0.015 val_acc=0.855 val_mF1=0.713
   ep070 loss=0.045 val_acc=0.790 val_mF1=0.665
   ep075 loss=0.145 val_acc=0.774 val_mF1=0.699
   early stop @ep75 (best @ep45)
   DONE acc=0.8699 macroF1=0.7153
L7_supcon | lora r=32 alpha=64 drop=0.0


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ADAPT=lora(r=32,a=64): trainable(LoRA) vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66
   ep000 loss=1.577 val_acc=0.468 val_mF1=0.369
   ep005 loss=0.233 val_acc=0.758 val_mF1=0.587
   ep010 loss=0.131 val_acc=0.855 val_mF1=0.758
   ep015 loss=0.087 val_acc=0.774 val_mF1=0.602
   ep020 loss=0.080 val_acc=0.823 val_mF1=0.673
   ep025 loss=0.136 val_acc=0.823 val_mF1=0.653
   ep030 loss=0.094 val_acc=0.839 val_mF1=0.670
   ep035 loss=0.082 val_acc=0.823 val_mF1=0.622
   ep040 loss=0.078 val_acc=0.823 val_mF1=0.670
   ep045 loss=0.077 val_acc=0.855 val_mF1=0.679
   ep050 loss=0.090 val_acc=0.839 val_mF1=0.658
   ep055 loss=0.078 val_acc=0.855 val_mF1=0.709
   ep060 loss=0.065 val_acc=0.871 val_mF1=0.754
   ep065 loss=0.113 val_acc=0.855 val_mF1=0.721
   ep070 loss=0.090 val_acc=0.871 val_mF1=0.750
   ep075 loss=0.096 val_acc=0.839 val_mF1=0.755
   ep080 loss=0.080 val_acc=0.823 val_mF1=0.715
   ep085 loss=0.083 val_acc=0.839 val_mF1=0.738
   ep090 loss=0.091 val_acc=0.